# BigSift — the minimal input that reproduces a failure

Provenance narrows a faulty output to the records that reached it. Delta debugging
then narrows *those* to the ones that actually cause the failure, by re-running the
query on subsets. The answer is in the **data**: which input records are to blame.

In [ ]:
import os, sys, glob

ROOT = os.environ.get("BIGASTERISK_HOME") or os.path.abspath("..")

# Jars: a source checkout has them under modules/*/target, the Docker image under jars/.
JARS = sorted(glob.glob(f"{ROOT}/modules/*/target/scala-2.13/bigasterisk-*.jar")) \
    or sorted(glob.glob(f"{ROOT}/jars/bigasterisk-*.jar"))
if not JARS:
    raise SystemExit("No BigAsterisk jars found. Run: bin/sbt package")

FASTUTIL_JAR = os.environ.get("FASTUTIL_JAR") or next(iter(sorted(
    glob.glob(f"{ROOT}/jars/fastutil*.jar")
    + glob.glob(os.path.expanduser("~/Library/Caches/Coursier/**/fastutil-8.5.15.jar"), recursive=True)
    + glob.glob(os.path.expanduser("~/.cache/coursier/**/fastutil-8.5.15.jar"), recursive=True)
)), None)
if not FASTUTIL_JAR:
    raise SystemExit("fastutil jar not found. Run: bin/sbt package")

SPARK_JARS = ",".join(JARS + [FASTUTIL_JAR])
DATA = f"{ROOT}/examples/data"
sys.path.insert(0, f"{ROOT}/python")

## The data

Twelve orders across three customers. One of them, `o8`, is an outlier at
`99999` — every notebook here uses it as the thing to find.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import bigasterisk

spark = (bigasterisk.configure(SparkSession.builder)
    .master("local[2]")
    .appName("bigsift-notebook")
    .config("spark.jars", SPARK_JARS)
    .config("spark.sql.adaptive.skewJoin.enabled", "false")
    .config("spark.ui.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

orders = spark.read.schema("oid STRING, cid STRING, amount INT").csv(f"{DATA}/orders.txt")
customers = spark.read.schema("cid STRING, name STRING").csv(f"{DATA}/customers.txt")
orders.createOrReplaceTempView("orders")
customers.createOrReplaceTempView("customers")

orders.show()

## A query with a faulty result

Amounts over 1000 are negated, so one group's total comes out negative.

In [ ]:
QUERY = ("SELECT cid, SUM(CASE WHEN amount > 1000 THEN -amount ELSE amount END) AS total "
    "FROM orders GROUP BY cid")
spark.sql(QUERY).show()

## Isolate the culprit rows

In [ ]:
result = bigasterisk.BigSift(spark).debug(
    "orders", QUERY, lambda r: r["total"] < 0)

print("provenance left", result.provenance_size, "candidates")
print("delta debugging narrowed them to", len(result.fault_inducing_rows))
for row in result.fault_inducing_rows:
    print(" ", row)

## Check

In [ ]:
assert len(result.fault_inducing_rows) == 1, result.fault_inducing_rows
assert result.fault_inducing_rows[0]["amount"] == 99999
print("OK")